# DQL Operator: Comparison Operators

Comparison operators filter numeric, date, and text values using greater-than and less-than logic.


## How to Think About Operators

Operators create true-or-false tests. Spark evaluates those tests for each row when filtering, joining, or building conditional expressions.

Before writing an operator expression, ask:

* Am I comparing one value to another?
* Am I checking membership in a list?
* Am I matching a text pattern?
* Am I checking a range?
* Do null values need special handling?

Good operator expressions are explicit and readable. For complex DataFrame filters, use `F.col(...)` so each column reference is clear.

## CSV Data Source

CSV stores data as plain text rows. The loader creates the table names used by the examples: `emp`, `dept`, `salgrade`, `projects`, and `emp_projects`.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "emp.csv").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find emp.csv. Put the CSV files in ./data or the current folder.")

print(f"Reading CSV files from: {DATA_DIR}")

emp = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp.csv"))
dept = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "dept.csv"))
salgrade = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "salgrade.csv"))
projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "projects.csv"))
emp_projects = spark.read.option("header", True).option("inferSchema", True).csv(str(DATA_DIR / "emp_projects.csv"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Parquet Data Source

Parquet stores data in a compressed columnar format. The same table names are created, so the DQL examples work the same way after loading Parquet.


In [ ]:
from pathlib import Path

from pyspark.sql import SparkSession
from pyspark.sql import functions as F
from pyspark.sql.window import Window

spark = (
    SparkSession.builder
    .appName("pyspark-parquet-sql-tutorial")
    .getOrCreate()
)

candidate_dirs = [
    Path.cwd() / "data",
    Path.cwd().parent / "data",
    Path.cwd(),
    Path.cwd().parent,
]

DATA_DIR = next((path for path in candidate_dirs if (path / "dept.parquet").exists()), None)
if DATA_DIR is None:
    raise FileNotFoundError("Could not find dept.parquet. Put the Parquet files in ./data or the current folder.")

emp_paths = sorted(DATA_DIR.glob("emp_part_*.parquet"))
if not emp_paths:
    raise FileNotFoundError("Could not find emp_part_*.parquet files in the Parquet data folder.")

print(f"Reading Parquet files from: {DATA_DIR}")

emp = spark.read.parquet(*[str(path) for path in emp_paths])
dept = spark.read.parquet(str(DATA_DIR / "dept.parquet"))
salgrade = spark.read.parquet(str(DATA_DIR / "salgrade.parquet"))
projects = spark.read.parquet(str(DATA_DIR / "projects.parquet"))
emp_projects = spark.read.parquet(str(DATA_DIR / "emp_projects.parquet"))

for name, df in {
    "emp": emp,
    "dept": dept,
    "salgrade": salgrade,
    "projects": projects,
    "emp_projects": emp_projects,
}.items():
    df.createOrReplaceTempView(name)


In [ ]:
emp.show(5)
dept.show()
salgrade.show()
projects.show()
emp_projects.show(5)


## Comparison Operators

Use greater than, greater than or equal, less than, and less than or equal for numeric and date comparisons.

Comparison operators are the building blocks of filters.

Plain-English meaning:

* `>` means greater than.
* `>=` means greater than or equal to.
* `<` means less than.
* `<=` means less than or equal to.


In [ ]:
emp.select("empno", "ename", "sal").where(F.col("sal") > 3000).show()
emp.select("empno", "ename", "sal").where(F.col("sal") >= 3000).show()
emp.select("empno", "ename", "sal").where(F.col("sal") < 1200).show()
emp.select("empno", "ename", "sal").where(F.col("sal") <= 1200).show()

spark.sql("""
SELECT empno, ename, sal FROM emp WHERE sal > 3000
""").show()
